In [7]:
import torch
import os
from torchvision import transforms
from PIL import Image
import json
from utils import rajouter_features

In [8]:
split = "train"  # choisir le split: "train", "val", "test"
crop_size = 32

# Création du dossier pour le split si besoin
os.makedirs(f"data/dataset_crops/{split}", exist_ok=True)

In [9]:
def get_features(img, img_burn):

    d = {}

    tensor = img.reshape(3, -1)
    pixels_values, pixels_nb = torch.unique(tensor, dim=1, return_counts=True)

    for classe in range(pixels_nb.shape[0]):
        valeur_pixel = pixels_values[:, classe]
        nb_pixels = pixels_nb[classe]

        if torch.equal(valeur_pixel, torch.tensor([0, 0, 0], dtype=torch.uint8)):
            d["feature0"] = nb_pixels.item()
        elif torch.equal(valeur_pixel, torch.tensor([57, 125, 73], dtype=torch.uint8)):
            d["feature1"] = nb_pixels.item()
        elif torch.equal(valeur_pixel, torch.tensor([65, 155, 223], dtype=torch.uint8)):
            d["feature2"] = nb_pixels.item()
        elif torch.equal(valeur_pixel, torch.tensor([122, 135, 198], dtype=torch.uint8)):
             d["feature3"] = nb_pixels.item()
        elif torch.equal(valeur_pixel, torch.tensor([136, 176, 83], dtype=torch.uint8)):
            d["feature4"] = nb_pixels.item()
        elif torch.equal(valeur_pixel, torch.tensor([165, 155, 143], dtype=torch.uint8)):
            d["feature5"] = nb_pixels.item()
        elif torch.equal(valeur_pixel, torch.tensor([196, 40, 27], dtype=torch.uint8)):
            d["feature6"] = nb_pixels.item()
        elif torch.equal(valeur_pixel, torch.tensor([223, 195, 90], dtype=torch.uint8)):
            d["feature7"] = nb_pixels.item()
        elif torch.equal(valeur_pixel, torch.tensor([228, 150, 53], dtype=torch.uint8)):
            d["feature8"] = nb_pixels.item()
        elif torch.equal(valeur_pixel, torch.tensor([168, 235, 255], dtype=torch.uint8)):
            d["feature9"] = nb_pixels.item()
        elif torch.equal(valeur_pixel, torch.tensor([97, 97, 97], dtype=torch.uint8)):
            d["feature10"] = nb_pixels.item()
        elif torch.equal(valeur_pixel, torch.tensor([13, 253, 21], dtype=torch.uint8)):
            d["feature1d1"] = nb_pixels.item()
        elif torch.equal(valeur_pixel, torch.tensor([165, 244, 167], dtype=torch.uint8)):
            d["feature1d2"] = nb_pixels.item()
        elif torch.equal(valeur_pixel, torch.tensor([65, 166, 68], dtype=torch.uint8)):
            d["feature1d3"] = nb_pixels.item()
        elif torch.equal(valeur_pixel, torch.tensor([16, 114, 20], dtype=torch.uint8)):
            d["feature1d4"] = nb_pixels.item()
        elif torch.equal(valeur_pixel, torch.tensor([242, 34, 214], dtype=torch.uint8)):
            d["feature7d1"] = nb_pixels.item()
        elif torch.equal(valeur_pixel, torch.tensor([207, 128, 253], dtype=torch.uint8)):
            d["feature7d2"] = nb_pixels.item()
        elif torch.equal(valeur_pixel, torch.tensor([145, 64, 206], dtype=torch.uint8)):
            d["feature7d3"] = nb_pixels.item()
        elif torch.equal(valeur_pixel, torch.tensor([84, 19, 121], dtype=torch.uint8)):
            d["feature7d4"] = nb_pixels.item()
        else:
            print("******* couleur non identifiée *********")
            print(valeur_pixel)
            print(nb_pixels)
            print(("**********************************"))

    if torch.equal(img_burn[:, 15, 15], torch.tensor([0, 0, 0], dtype=torch.uint8)):   # no visible damage
        d["damage"] = 0
    elif torch.equal(img_burn[:, 15, 15], torch.tensor([181, 254, 142], dtype=torch.uint8)):   # negligible to slight damage
        d["damage"] = 1
    elif torch.equal(img_burn[:, 15, 15], torch.tensor([254, 217, 142], dtype=torch.uint8)):   # moderate damage
        d["damage"] = 2
    elif torch.equal(img_burn[:, 15, 15], torch.tensor([254, 153, 41], dtype=torch.uint8)):   # high damage
        d["damage"] = 3
    elif torch.equal(img_burn[:, 15, 15], torch.tensor([204, 76, 2], dtype=torch.uint8)):   # destruction
        d["damage"] = 4
    elif torch.equal(img_burn[:, 15, 15], torch.tensor([255, 255, 255], dtype=torch.uint8)):   # problème de couleur avec le 134e masque de brûlé
        d["damage"] = 0   # après inspection visuelle correspond à un no visible damage
    else:
        print("********** niveau de brulé non identifié *********")
        print(img_burn[:, 15, 15])
        print(("**********************************"))

    return d

In [10]:
def get_feature_vector(donnes):
    # exemple de donnes : {'feature1': 736, 'feature7': 288, 'damage': 0} 

    features = ["feature1", "feature2", "feature3", "feature4", "feature6", "feature7", "feature8", "feature1d1", "feature1d2", "feature1d3", "feature1d4", "feature7d1", "feature7d2", "feature7d3", "feature7d4"]
    nb_intervals = 4
    boundaries = torch.linspace(0, 1024, nb_intervals + 1)[1:-1]  # ignore les extrémités

    d_temp = {}

    for feature in features:
        coverage = donnes.get(feature, None)

        if coverage is not None:
            one_hot = [0] * nb_intervals
            for i, threshold in enumerate(boundaries):
                if coverage < threshold:
                    one_hot[i] = 1
                    break
            else:
                # si coverage >= à tous les seuils
                one_hot[-1] = 1
        else:
            one_hot = [0] * nb_intervals

        d_temp[feature] = one_hot

    x = [e for feature in features for e in d_temp[feature]]
    y = donnes.get("damage", None)

    return x, y

In [11]:
dico = {}
# dico = {crop_0: {"chemin de l'image associée": chemin_img, "chemin de l'image de brûlé associée": chemin_img_burn, "coordonnées du crop": (start_h, start_w), "attributs": x, "niveau de brûlé": y}, crop_1: {...}, ...}
idx = 0

transform = transforms.ToTensor()

cpt = 0

for (root, dirs, files) in os.walk(f"data/dataset/{split}"):
    for file in files:
        if "Esri10_LC" in file and (file.endswith(".png")):

            cpt += 1
            print(cpt)

            # charger l'image annotées
            path_to_img = os.path.join(root, file)
            img = Image.open(path_to_img).convert("RGB")

            # charger l'image satellite
            path_to_img_sat = path_to_img.replace("Esri10_LC", "S2L2A")
            img_sat = Image.open(path_to_img_sat).convert("RGB")
            tensor_sat = transform(img_sat)

            # charger le masque de brûlé
            path_to_img_gra = path_to_img.replace("Esri10_LC", "GRA")
            path_to_img_del = path_to_img.replace("Esri10_LC", "DEL")
            if os.path.exists(path_to_img_gra):
                path_to_img_burn = path_to_img_gra
            elif os.path.exists(path_to_img_del):
                path_to_img_burn = path_to_img_del
            else:
                print("********** image brûlée non trouvée *********")
                continue
            img_burn = Image.open(path_to_img_burn).convert("RGB")
            tensor_burn = (transform(img_burn) * 255).byte()

            # rajouter les features artificielles
            img = rajouter_features(img, img_burn)
            tensor = (transform(img) * 255).byte()


            # Obtenir les dimensions du tenseur (C, H, W)
            channels, height, width = tensor.shape
            
            # Calculer le nombre de carrés possibles
            num_crops_h = height // crop_size
            num_crops_w = width // crop_size

            # Découper l'image en carrés de 32x32
            for i in range(num_crops_h):
                for j in range(num_crops_w):
                    # Calculer les coordonnées de début
                    start_h = i * crop_size
                    start_w = j * crop_size
                    end_h = start_h + crop_size
                    end_w = start_w + crop_size

                    # Extraire le crop
                    crop = tensor[:, start_h:end_h, start_w:end_w]
                    crop_sat = tensor_sat[:, start_h:end_h, start_w:end_w]
                    crop_burn = tensor_burn[:, start_h:end_h, start_w:end_w]

                    # Indicer le crop
                    dico[f"crop_{idx}"] = {}

                    # Calculer les features du crop
                    features = get_features(crop, crop_burn)

                    # obtenir le vecteur à partir des features
                    x, y = get_feature_vector(features)

                    # remplir le dico
                    dico[f"crop_{idx}"]["chemin de l'image associée"] = path_to_img
                    dico[f"crop_{idx}"]["chemin de l'image de brûlé associée"] = path_to_img_burn
                    dico[f"crop_{idx}"]["coordonnées du crop"] = (start_h, start_w)
                    dico[f"crop_{idx}"]["attributs"] = x
                    dico[f"crop_{idx}"]["niveau de brûlé"] = y

                    # enregistrer l'image
                    crop = transforms.ToPILImage()(crop_sat)
                    crop.save(os.path.join(f"data/dataset_crops/{split}", f"crop_{idx}.png"))

                    idx += 1


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277


In [12]:
with open(f'data/dataset_crops/{split}_{crop_size}x{crop_size}_data.json', 'w', encoding='utf-8') as f:
    json.dump(dico, f, ensure_ascii=False, indent=4)